# 05 · RoBERTa fine-tuning for Empathy prediction (Days 11-13)

Fine-tune RoBERTa-base on the WASSA CONV-Turn Empathy target and compare against
the Ridge + TF-IDF baseline established in `04_tfidf_sweeps.ipynb`.

**Approach.** RoBERTa (Liu et al., 2019) is a transformer-based language model
pretrained on ~160GB of English text using a masked language modeling objective.
Unlike the bag-of-words TF-IDF representation used in the baseline, RoBERTa
produces contextualized token embeddings that capture word meaning in context.
We adapt the pretrained model to Empathy prediction by attaching a regression
head to the `[CLS]` token output and fine-tuning all parameters on the training
split.

**Setup.** Fine-tuning is orchestrated via Hugging Face's `Trainer` API, which
handles the training loop, evaluation, checkpointing, and metric computation.
Hyperparameters follow standard practice for BERT-family fine-tuning:
learning rate 2e-5, batch size 16, 3 epochs, AdamW optimizer with linear warmup.
The checkpoint with the highest development-set Pearson correlation is retained
as the final model.

**Split note.** This notebook uses the same internal conversation-grouped
70/15/15 split as the baseline notebooks, kept for consistency across the
Ridge–RoBERTa comparison. All model-selection decisions are made on dev; test
is used only for final reporting.

In [1]:
!git clone https://github.com/DavorSopar/thesis-empathy.git /content/thesis-empathy
import sys
sys.path.insert(0, '/content/thesis-empathy/src')

Cloning into '/content/thesis-empathy'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (127/127), done.
remote: Total 133 (delta 55), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 2.05 MiB | 2.08 MiB/s, done.
Resolving deltas: 100% (55/55), done.


In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
# Make src/ importable whether run from notebooks/ or the repo root.
REPO_ROOT = Path.cwd()
if (REPO_ROOT / "src").is_dir():
    pass
elif (REPO_ROOT.parent / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
from data import load_convt, impute_selfdisclosure, TARGETS

RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42

## 1 · Load the three official WASSA splits

The WASSA-released training file contains all conversations from the
underlying corpus, including those subsequently released as development
and test splits. We filter the training set to exclude any conversation
appearing in dev or test, yielding a conversation-disjoint partition.


In [3]:
# Load all three splits
train_raw = load_convt('train')
dev = load_convt('dev')
test = load_convt('test')

# Filter train: remove any conversation appearing in dev or test
excluded_ids = set(dev.conversation_id) | set(test.conversation_id)
train = train_raw[~train_raw.conversation_id.isin(excluded_ids)].reset_index(drop=True)

# Sanity check: no conversation overlap between splits
assert not (set(train.conversation_id) & set(dev.conversation_id))
assert not (set(train.conversation_id) & set(test.conversation_id))
assert not (set(dev.conversation_id) & set(test.conversation_id))

split_tbl = pd.DataFrame({
    'turns':         [len(train), len(dev), len(test)],
    'conversations': [train.conversation_id.nunique(),
                      dev.conversation_id.nunique(),
                      test.conversation_id.nunique()],
}, index=['train', 'dev', 'test'])
print(split_tbl)
print('\nAll three splits are conversation-disjoint.')


       turns  conversations
train   9330            405
dev      990             33
test    2061             63

All three splits are conversation-disjoint.


## 2 · Metric helpers


In [4]:
def pearson(y_true, y_pred):
    """Pearson r; returns NaN when either side is constant (undefined)."""
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    if np.std(y_true) < 1e-12 or np.std(y_pred) < 1e-12:
        return np.nan
    return float(np.corrcoef(y_true, y_pred)[0, 1])

def score_all(y_true, y_pred):
    return {
        "pearson": pearson(y_true, y_pred),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }

## 1 · Preprocessing — tokenization and dataset formatting

RoBERTa expects input as sequences of subword token IDs from its fixed
pretrained vocabulary (~50,000 subwords). Text preprocessing consists of two
steps: (i) tokenize each turn into `input_ids` and `attention_mask` using
RoBERTa's matched pretrained tokenizer, and (ii) convert the pandas DataFrames
into Hugging Face `Dataset` objects with the target column renamed to `labels`
(the field name expected by `Trainer`). Sequences are truncated at 128 tokens,
which covers essentially all conversation turns in the WASSA data without
padding overhead.

In [5]:
from transformers import AutoTokenizer
from datasets import Dataset

checkpoint = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

# Convert pandas → HF Dataset, rename Empathy to labels (Trainer expects "labels")
train_ds = Dataset.from_pandas(train[["text", "Empathy"]].rename(columns={"Empathy": "labels"}))
dev_ds   = Dataset.from_pandas(dev[["text", "Empathy"]].rename(columns={"Empathy": "labels"}))
test_ds  = Dataset.from_pandas(test[["text", "Empathy"]].rename(columns={"Empathy": "labels"}))

# Tokenize all splits
train_ds = train_ds.map(tokenize_fn, batched=True)
dev_ds   = dev_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

# Ensure labels are float32 (regression needs floats, not ints)
import torch
train_ds = train_ds.map(lambda x: {"labels": float(x["labels"])})
dev_ds   = dev_ds.map(lambda x: {"labels": float(x["labels"])})
test_ds  = test_ds.map(lambda x: {"labels": float(x["labels"])})

# Sanity check
print(train_ds[0])

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

Map:   0%|          | 0/2061 [00:00<?, ? examples/s]

Map:   0%|          | 0/9330 [00:00<?, ? examples/s]

Map:   0%|          | 0/990 [00:00<?, ? examples/s]

Map:   0%|          | 0/2061 [00:00<?, ? examples/s]

{'text': 'what did you think about this article', 'labels': 0.6667, 'input_ids': [0, 12196, 222, 47, 206, 59, 42, 1566, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [19]:
print(f"dev DataFrame size: {len(dev)}, first text: {dev['text'].iloc[0][:60]}")
print(f"test DataFrame size: {len(test)}, first text: {test['text'].iloc[0][:60]}")
print()
print(f"dev_ds size: {len(dev_ds)}, first text: {dev_ds[0]['text'][:60]}")
print(f"test_ds size: {len(test_ds)}, first text: {test_ds[0]['text'][:60]}")

dev DataFrame size: 990, first text: Hello how are you?
test DataFrame size: 2061, first text: Yeah, I'm sorry but celebrity life doesn't interest me that 

dev_ds size: 990, first text: Hello how are you?
test_ds size: 2061, first text: Yeah, I'm sorry but celebrity life doesn't interest me that 


In [6]:
print(f"train: {len(train_ds)}, dev: {len(dev_ds)}, test: {len(test_ds)}")

train: 9330, dev: 990, test: 2061


## 2 · Model and Trainer configuration

The model is loaded via `AutoModelForSequenceClassification.from_pretrained`
with `num_labels=1` and `problem_type="regression"`, which attaches a
randomly-initialized regression head on top of the pretrained transformer body.
During fine-tuning, both the pretrained body and the new head are updated: the
head learns from scratch (starting from random weights), while the body is
gently adjusted from its pretrained state by using a small learning rate (2e-5).
This preserves the general language understanding acquired during pretraining
while adapting the model's outputs to the empathy prediction task.

`DataCollatorWithPadding` handles per-batch dynamic padding, avoiding the waste
of padding all sequences to a fixed dataset-wide maximum. A custom
`compute_metrics` function reports MAE, RMSE, and Pearson r after each
evaluation pass; the best checkpoint is selected by Pearson correlation on the
development set.

In [15]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr
import numpy as np

# Load the model with a regression head
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=1,
    problem_type="regression",
)

# Data collator handles per-batch padding automatically
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Metrics function — same three you've been using
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.squeeze()  # shape (n, 1) → (n,)
    mae = mean_absolute_error(labels, predictions)
    rmse = np.sqrt(mean_squared_error(labels, predictions))
    r = pearsonr(labels, predictions)[0]
    return {"mae": mae, "rmse": rmse, "pearson": r}

# Training configuration
training_args = TrainingArguments(
    output_dir="./roberta_empathy",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="pearson",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",  # disables wandb/tensorboard reporting
)

# Assemble the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Sanity check — this should print without errors
print("Trainer is ready. Do not call trainer.train() yet.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainer is ready. Do not call trainer.train() yet.


## 3 · Training

Fine-tuning runs for three epochs on the training split, with evaluation on the
development split after each epoch. Both training loss and development metrics
are logged to monitor for overfitting.

In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss,Mae,Rmse,Pearson
1,0.489083,0.751497,0.692647,0.866889,0.619060
2,0.349945,0.745660,0.692995,0.863516,0.622137


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1168, training_loss=0.47169036897894456, metrics={'train_runtime': 355.299, 'train_samples_per_second': 52.519, 'train_steps_per_second': 3.287, 'total_flos': 606202428653568.0, 'train_loss': 0.47169036897894456, 'epoch': 2.0})

## 4 · Final evaluation

Evaluate the best-checkpoint model on both dev (as a reference) and test (the
number reported in the thesis Results section). Test is held out and touched
only once, at this stage.

In [17]:
# Evaluate on dev
dev_results = trainer.evaluate(dev_ds)
print("DEV:", dev_results)

# Evaluate on test
test_results = trainer.evaluate(test_ds)
print("TEST:", test_results)

Training Loss,Validation Loss,Epoch,Mae,Rmse,Pearson
0.349945,0.745660,2,0.692995,0.863516,0.622137


DEV: {'eval_loss': 0.7456603646278381, 'eval_mae': 0.6929948925971985, 'eval_rmse': 0.8635162793067876, 'eval_pearson': 0.6221365332603455}


Training Loss,Validation Loss,Epoch,Mae,Rmse,Pearson
0.349945,1.200267,2,0.870488,1.095567,0.558478


TEST: {'eval_loss': 1.2002671957015991, 'eval_mae': 0.8704879879951477, 'eval_rmse': 1.0955670658164196, 'eval_pearson': 0.5584781765937805}


In [13]:
# Save the fine-tuned RoBERTa model and tokenizer
SAVE_DIR = "./roberta_empathy_final"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"Saved model + tokenizer to {SAVE_DIR}/")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model + tokenizer to ./roberta_empathy_final/


In [14]:
import pandas as pd

# Save the evaluation metrics for the thesis Results section
results_df = pd.DataFrame([
    {"model": "RoBERTa-base", "split": "dev",  **dev_results},
    {"model": "RoBERTa-base", "split": "test", **test_results},
])

results_df.to_csv("roberta_empathy_final_results.csv", index=False)
print("Saved: roberta_empathy_final_results.csv")
print(results_df)

Saved: roberta_empathy_final_results.csv
          model split  eval_loss  eval_mae  eval_rmse  eval_pearson
0  RoBERTa-base   dev     0.7785    0.7059     0.8823        0.6031
1  RoBERTa-base  test     1.2089    0.8699     1.0995        0.5571


In [18]:
# Are dev and test being processed the same way?
print(f"Dev size: {len(test_ds)}")
print(f"Test size: {len(dev_ds)}")
print(f"Dev sample: {dev_ds[0]}")
print(f"Test sample: {test_ds[0]}")

# Distribution of empathy in each
import numpy as np
print(f"\nDev empathy: mean={np.mean(dev['Empathy']):.3f}, std={np.std(dev['Empathy']):.3f}")
print(f"Test empathy: mean={np.mean(test['Empathy']):.3f}, std={np.std(test['Empathy']):.3f}")

Dev size: 2061
Test size: 990
Dev sample: {'text': 'Hello how are you?', 'labels': 1.0, 'input_ids': [0, 31414, 141, 32, 47, 116, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}
Test sample: {'text': "Yeah, I'm sorry but celebrity life doesn't interest me that much. I don't know what to think or feel about this article.", 'labels': 1.0, 'input_ids': [0, 14783, 6, 38, 437, 6661, 53, 6794, 301, 630, 75, 773, 162, 14, 203, 4, 38, 218, 75, 216, 99, 7, 206, 50, 619, 59, 42, 1566, 4, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Dev empathy: mean=2.077, std=1.098
Test empathy: mean=2.484, std=1.257
